# Proyecto: Detección de Fraude en Seguros Vehiculares
### Análisis Exploratorio, Modelado Estadístico y Machine Learning

**Autores:** Andres Felipe Londoño Ocampo - Jhonatan Caro Atehortua - Paulina Perez Ramirez  - Victor Manuerl Galeano Alvarez   
**Curso:** Proyecto 1
**Profesor:** David Palacio Jimenez 
**Universidad de Medellín**
**05/2026**

---

## Resumen 

Este notebook desarrolla un análisis completo del dataset **Vehicle Insurance Claim Fraud Detection** (Oracle), con el objetivo de identificar reclamaciones de seguros vehiculares fraudulentas. Se aplican técnicas de **análisis exploratorio de datos (EDA)**, **preprocesamiento con Pipelines de Scikit-Learn**, y en este se entrenan **cuatro modelos de clasificación** los cuales son: Regresión Logística, Random Forest, XGBoost y un modelo con balanceo SMOTE. Se evalúa la interpretabilidad mediante **coeficientes, importancia de variables, permutation importance y SHAP values**, y finalmente se presenta una **aplicación en Streamlit** para uso interactivo del modelo.


## 1. Comprensión del problema

### 1.1 Origen de los datos

El dataset **`fraud_oracle.csv`** proviene del repositorio público de **Oracle / Kaggle** (*Vehicle Insurance Claim Fraud Detection*) y contiene **15,420 reclamaciones de seguros vehiculares** registradas entre los años **1994 y 1996** en Norteamérica. Los datos fueron recolectados por una compañía aseguradora con fines de auditoría y soporte a investigaciones internas de fraude.

### 1.2 Problemática

El **fraude en seguros vehiculares** representa pérdidas multimillonarias para la industria aseguradora a nivel mundial y, de manera indirecta, encarece las primas que pagan los asegurados honestos. La detección manual de reclamaciones fraudulentas es lenta, costosa y propensa a errores humanos.

### 1.3 Contexto de negocio

La aseguradora necesita un **sistema de soporte a la decisión** que asigne una *probabilidad de fraude* a cada reclamación nueva, de modo que los analistas puedan **priorizar la revisión** de los casos de mayor riesgo. Un modelo predictivo bien calibrado:

- **Reduce costos operativos** (menos casos a revisar manualmente).
- **Aumenta la tasa de detección** (recall) sin generar fricción excesiva con clientes legítimos.
- **Genera trazabilidad** para auditorías y cumplimiento regulatorio.


## 2. Descripción general de la información

El dataset contiene **33 variables** que describen tres dimensiones de cada reclamación:

| Dimensión | Variables principales |
|-----------|----------------------|
| **Temporal** | `Month`, `WeekOfMonth`, `DayOfWeek`, `MonthClaimed`, `WeekOfMonthClaimed`, `DayOfWeekClaimed`, `Year` |
| **Vehículo y póliza** | `Make`, `VehicleCategory`, `VehiclePrice`, `AgeOfVehicle`, `PolicyType`, `BasePolicy`, `Deductible`, `NumberOfCars` |
| **Asegurado y siniestro** | `Sex`, `MaritalStatus`, `Age`, `AgeOfPolicyHolder`, `Fault`, `AccidentArea`, `PoliceReportFiled`, `WitnessPresent`, `PastNumberOfClaims`, `AddressChange_Claim`, `Days_Policy_Accident`, `Days_Policy_Claim`, `NumberOfSuppliments`, `AgentType`, `RepNumber`, `DriverRating` |
| **Identificador / target** | `PolicyNumber`, **`FraudFound_P`** (target) |

### 2.1 Diccionario de variables

| Variable | Descripción | Tipo de dato | Tipo estadístico |
|----------|-------------|--------------|------------------|
| `Month` | Mes en que ocurrió el accidente | object | Categórica nominal |
| `WeekOfMonth` | Semana del mes del accidente (1–5) | int | Discreta ordinal |
| `DayOfWeek` | Día de la semana del accidente | object | Categórica nominal |
| `Make` | Marca del vehículo | object | Categórica nominal |
| `AccidentArea` | Zona del accidente (Urban / Rural) | object | Binaria |
| `DayOfWeekClaimed` | Día de la semana en que se reclamó | object | Categórica nominal |
| `MonthClaimed` | Mes en que se reclamó | object | Categórica nominal |
| `WeekOfMonthClaimed` | Semana del mes de la reclamación | int | Discreta ordinal |
| `Sex` | Sexo del asegurado | object | Binaria |
| `MaritalStatus` | Estado civil | object | Categórica nominal |
| `Age` | Edad del asegurado (años) | int | Continua |
| `Fault` | Responsable del accidente | object | Binaria |
| `PolicyType` | Tipo de póliza completo | object | Categórica nominal |
| `VehicleCategory` | Categoría del vehículo | object | Categórica nominal |
| `VehiclePrice` | Rango de precio del vehículo (USD) | object | Ordinal |
| **`FraudFound_P`** | ** Target: 1 = fraude, 0 = legítimo** | int | Binaria |
| `PolicyNumber` | ID de la póliza | int | Categórica nominal |
| `RepNumber` | ID del representante de la aseguradora | int | Categórica nominal |
| `Deductible` | Deducible de la póliza (USD) | int | Continua |
| `DriverRating` | Calificación del conductor (1–4) | int | Ordinal |
| `Days_Policy_Accident` | Días entre inicio de póliza y accidente | object | Ordinal |
| `Days_Policy_Claim` | Días entre inicio de póliza y reclamación | object | Ordinal |
| `PastNumberOfClaims` | Reclamaciones previas del asegurado | object | Ordinal |
| `AgeOfVehicle` | Antigüedad del vehículo | object | Ordinal |
| `AgeOfPolicyHolder` | Rango de edad del asegurado | object | Ordinal |
| `PoliceReportFiled` | ¿Se presentó reporte policial? | object | Binaria |
| `WitnessPresent` | ¿Hubo testigos? | object | Binaria |
| `AgentType` | Agente que tramita (Internal / External) | object | Binaria |
| `NumberOfSuppliments` | Número de suplementos a la reclamación | object | Ordinal |
| `AddressChange_Claim` | Tiempo desde cambio de dirección | object | Ordinal |
| `NumberOfCars` | Número de vehículos del asegurado | object | Ordinal |
| `Year` | Año del accidente | int | Discreta |
| `BasePolicy` | Tipo base de la póliza | object | Categórica nominal |


### 2.2 Objetivo

Construir un **modelo de clasificación binaria** capaz de predecir si una reclamación de seguro vehicular es fraudulenta, con énfasis en **maximizar el recall de la clase minoritaria** (fraudes) sin sacrificar excesivamente la precisión.

### Preguntas

1. ¿Qué **perfil sociodemográfico** (sexo, edad, estado civil) está más asociado a fraude?
2. ¿Qué **tipo de póliza, vehículo y siniestro** muestran mayor incidencia de fraude?
3. ¿Existen **patrones temporales** (mes, día, semana) en las reclamaciones fraudulentas?
4. ¿El **historial del asegurado** (reclamaciones previas, cambio de dirección, antigüedad de la póliza) predice fraude?
5. ¿Qué **modelo** ofrece el mejor compromiso entre detección, precisión e interpretabilidad?


## 3. Configuración del entorno

In [72]:
"""Se importan todas las librerías necesarias y se fijan las semillas para reproducibilidad."""

# Manipulación y análisis
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Visualización (Plotly exclusivamente como pide el enunciado) 
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "plotly_white"

# Scikit-learn: preprocesamiento y modelado
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)

# XGBoost
from xgboost import XGBClassifier

# Manejo del desbalance
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Interpretabilidad
import shap

# Reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## 4. Carga de datos

In [73]:
# Cargar el dataset
df = pd.read_csv('../data/raw/fraud_oracle.csv')

print(f"Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Memoria: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Duplicados: {df.duplicated().sum()}")
print(f"Nulos: {df.isnull().sum().sum()}")
df.head()


Dimensiones: 15,420 filas × 33 columnas
Memoria: 23.85 MB
Duplicados: 0
Nulos: 0


,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,...,3 years,26 to 30,No,No,External,none,1 year,3 to 4,1994,Liability
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,...,6 years,31 to 35,Yes,No,External,none,no change,1 vehicle,1994,Collision
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,...,7 years,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,...,more than 7,51 to 65,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,...,5 years,31 to 35,No,No,External,none,no change,1 vehicle,1994,Collision


In [74]:
# Tipos de datos
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15420 entries, 0 to 15419
Data columns (total 33 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Month                 15420 non-null  object
 1   WeekOfMonth           15420 non-null  int64 
 2   DayOfWeek             15420 non-null  object
 3   Make                  15420 non-null  object
 4   AccidentArea          15420 non-null  object
 5   DayOfWeekClaimed      15420 non-null  object
 6   MonthClaimed          15420 non-null  object
 7   WeekOfMonthClaimed    15420 non-null  int64 
 8   Sex                   15420 non-null  object
 9   MaritalStatus         15420 non-null  object
 10  Age                   15420 non-null  int64 
 11  Fault                 15420 non-null  object
 12  PolicyType            15420 non-null  object
 13  VehicleCategory       15420 non-null  object
 14  VehiclePrice          15420 non-null  object
 15  FraudFound_P          15420 non-null

In [75]:
# Distribución de la variable objetivo
target_counts = df['FraudFound_P'].value_counts()
target_pct = df['FraudFound_P'].value_counts(normalize=True).mul(100).round(2)

print("Distribución de la variable objetivo (FraudFound_P):")
print(pd.DataFrame({'Conteo': target_counts, 'Porcentaje (%)': target_pct}))
print(f"\n  Ratio de desbalance: 1 fraude por cada {target_counts[0]/target_counts[1]:.1f} casos legítimos.")

# Visualización
fig = px.pie(
    values=target_counts.values,
    names=['Legítimo (0)', 'Fraude (1)'],
    title='Distribución de la variable objetivo: FraudFound_P',
    color_discrete_sequence=['#2E86AB', '#E63946'],
    hole=0.4
)
fig.update_traces(textinfo='percent+label+value')
fig.show()


Distribución de la variable objetivo (FraudFound_P):
              Conteo  Porcentaje (%)
FraudFound_P                        
0              14497           94.01
1                923            5.99

  Ratio de desbalance: 1 fraude por cada 15.7 casos legítimos.


## 5. Tratamiento preliminar de datos

1. **Detección de valores faltantes encubiertos**  como por ejemplo `Age=0`,  ó `'0'` en variables como mes/día reclamado.
2. **Imputación**: mediana para variables numéricas y moda para variables categóricas.
3. **Conversión de variables ordinales textuales** a valores numéricos ordinales.
4. **Eliminación de columnas no informativas**  como por ejemplo `PolicyNumber` (ID de la póliza).


In [76]:
# 5.1 Detección de valores faltantes encubiertos 
print(" VALORES FALTANTES ENCUBIERTOS \n")
print(f"• Age = 0  →  {(df['Age']==0).sum()} registros ({(df['Age']==0).mean()*100:.2f}%)")
print(f"• MonthClaimed = '0'      →  {(df['MonthClaimed']=='0').sum()} registros")
print(f"• DayOfWeekClaimed = '0'  →  {(df['DayOfWeekClaimed']=='0').sum()} registros")


 VALORES FALTANTES ENCUBIERTOS 

• Age = 0  →  320 registros (2.08%)
• MonthClaimed = '0'      →  1 registros
• DayOfWeekClaimed = '0'  →  1 registros


In [77]:
# 5.2 Imputación de valores faltantes 
df_clean = df.copy()

# Age = 0 → mediana de las edades válidas (>0)
median_age = int(df_clean.loc[df_clean['Age'] > 0, 'Age'].median())
df_clean.loc[df_clean['Age'] == 0, 'Age'] = median_age
print(f"Age = 0 imputada con la mediana = {median_age} años")

# MonthClaimed = '0' → moda
mode_month = df_clean.loc[df_clean['MonthClaimed'] != '0', 'MonthClaimed'].mode()[0]
df_clean.loc[df_clean['MonthClaimed'] == '0', 'MonthClaimed'] = mode_month
print(f"MonthClaimed = '0' imputado con la moda = '{mode_month}'")

# DayOfWeekClaimed = '0' → moda
mode_day = df_clean.loc[df_clean['DayOfWeekClaimed'] != '0', 'DayOfWeekClaimed'].mode()[0]
df_clean.loc[df_clean['DayOfWeekClaimed'] == '0', 'DayOfWeekClaimed'] = mode_day
print(f"DayOfWeekClaimed = '0' imputado con la moda = '{mode_day}'")


Age = 0 imputada con la mediana = 39 años
MonthClaimed = '0' imputado con la moda = 'Jan'
DayOfWeekClaimed = '0' imputado con la moda = 'Monday'


In [78]:
"""
5.3 Mapeo de variables ordinales textuales a numéricas 
con las siguientes variables hacemos un mapeo, toda vez que son variables ordinales, es decir, tienen un orden natural.
 """

ordinal_mappings = {
    'VehiclePrice': {
        'less than 20000': 0, '20000 to 29000': 1, '30000 to 39000': 2,
        '40000 to 59000': 3, '60000 to 69000': 4, 'more than 69000': 5
    },
    'Days_Policy_Accident': {
        'none': 0, '1 to 7': 1, '8 to 15': 2, '15 to 30': 3, 'more than 30': 4
    },
    'Days_Policy_Claim': {
        'none': 0, '8 to 15': 1, '15 to 30': 2, 'more than 30': 3
    },
    'PastNumberOfClaims': {
        'none': 0, '1': 1, '2 to 4': 2, 'more than 4': 3
    },
    'AgeOfVehicle': {
        'new': 0, '2 years': 1, '3 years': 2, '4 years': 3,
        '5 years': 4, '6 years': 5, '7 years': 6, 'more than 7': 7
    },
    'NumberOfSuppliments': {
        'none': 0, '1 to 2': 1, '3 to 5': 2, 'more than 5': 3
    },
    'AddressChange_Claim': {
        'no change': 0, 'under 6 months': 1, '1 year': 2, '2 to 3 years': 3, '4 to 8 years': 4
    },
    'NumberOfCars': {
        '1 vehicle': 1, '2 vehicles': 2, '3 to 4': 3, '5 to 8': 4, 'more than 8': 5
    }
}

for col, mapping in ordinal_mappings.items():
    df_clean[col] = df_clean[col].map(mapping).astype(int)

print("Variables ordinales convertidas a numéricas:")
for col in ordinal_mappings:
    print(f"   • {col}: rango [{df_clean[col].min()}, {df_clean[col].max()}]")


Variables ordinales convertidas a numéricas:
   • VehiclePrice: rango [0, 5]
   • Days_Policy_Accident: rango [0, 4]
   • Days_Policy_Claim: rango [0, 3]
   • PastNumberOfClaims: rango [0, 3]
   • AgeOfVehicle: rango [0, 7]
   • NumberOfSuppliments: rango [0, 3]
   • AddressChange_Claim: rango [0, 4]
   • NumberOfCars: rango [1, 5]


In [79]:
"""
5.4 elimarmos el ID de la póliza, ya que es un identificador único sin poder predictivo,
el año porque es una variable que no se repitira y no aporta información relevante para el modelo
y el rango de la edad del asegurado porque ya tenemos una variable que es la edad real del cliente
"""
df_clean = df_clean.drop(columns=['PolicyNumber', 'Year','AgeOfPolicyHolder'])

In [80]:
# 5.5 Análisis de outliers, variable Age
q1, q3 = df_clean['Age'].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
print(f"Rango aceptable: [{lower:.0f}, {upper:.0f}]")
print(f"Rango observado: [{df_clean['Age'].min()}, {df_clean['Age'].max()}]")


"""
Se decide no retirar los valores hasta 80 años por la naturaleza del dataset (seguros de autos),
es posible que existan asegurados de 80 años, de igual manera tampoco se retiran valores inferiores a 18 años, 
toda vez que al realizar un berev investigacion en EEUU, se observa que la edad mínima para conducir es de 16 años, 
y es posible que existan asegurados jóvenes, eliminar esos registros podría sesgar el análisis, 
por lo tanto, se opta por conservar todos los datos de la variable Age, incluyendo los considerados outliers por la regla IQR.
"""


Rango aceptable: [6, 74]
Rango observado: [16, 80]


'\nSe decide no retirar los valores hasta 80 años por la naturaleza del dataset (seguros de autos),\nes posible que existan asegurados de 80 años, de igual manera tampoco se retiran valores inferiores a 18 años, \ntoda vez que al realizar un berev investigacion en EEUU, se observa que la edad mínima para conducir es de 16 años, \ny es posible que existan asegurados jóvenes, eliminar esos registros podría sesgar el análisis, \npor lo tanto, se opta por conservar todos los datos de la variable Age, incluyendo los considerados outliers por la regla IQR.\n'

## 6. Análisis descriptivo

In [81]:
# 6.1 Medidas de tendencia central y dispersión (variables numéricas) 
numeric_cols = df_clean.select_dtypes(include=np.number).columns.drop('FraudFound_P')
desc = df_clean[numeric_cols].describe().T
desc['range'] = desc['max'] - desc['min']
desc['cv'] = (desc['std'] / desc['mean']).round(3)  # coeficiente de variación
desc.round(2)


,count,mean,std,min,25%,50%,75%,max,range,cv
WeekOfMonth,15420.0,2.79,1.29,1.0,2.0,3.0,4.0,5.0,4.0,0.46
WeekOfMonthClaimed,15420.0,2.69,1.26,1.0,2.0,3.0,4.0,5.0,4.0,0.47
Age,15420.0,40.67,12.18,16.0,31.0,39.0,48.0,80.0,64.0,0.30
VehiclePrice,15420.0,1.80,1.44,0.0,1.0,1.0,2.0,5.0,5.0,0.80
RepNumber,15420.0,8.48,4.60,1.0,5.0,8.0,12.0,16.0,15.0,0.54
Deductible,15420.0,407.70,43.95,300.0,400.0,400.0,400.0,700.0,400.0,0.11
DriverRating,15420.0,2.49,1.12,1.0,1.0,2.0,3.0,4.0,3.0,0.45
Days_Policy_Accident,15420.0,3.97,0.29,0.0,4.0,4.0,4.0,4.0,4.0,0.07
Days_Policy_Claim,15420.0,2.99,0.10,0.0,3.0,3.0,3.0,3.0,3.0,0.03
PastNumberOfClaims,15420.0,1.33,1.02,0.0,0.0,1.0,2.0,3.0,3.0,0.77


In [82]:
# 6.2 Tasa de fraude por variable categórica clave 
def fraud_rate_by(col):
    g = df_clean.groupby(col)['FraudFound_P'].agg(['count', 'sum', 'mean']).round(4)
    g.columns = ['n_total', 'n_fraudes', 'tasa_fraude']
    g['tasa_fraude_%'] = (g['tasa_fraude'] * 100).round(2)
    return g.sort_values('tasa_fraude', ascending=False)

print("Tasa de fraude por TIPO DE PÓLIZA BASE:\n")
print(fraud_rate_by('BasePolicy'))
print("\nTasa de fraude por RESPONSABLE (Fault):\n")
print(fraud_rate_by('Fault'))
print("\nTasa de fraude por ZONA DE ACCIDENTE:\n")
print(fraud_rate_by('AccidentArea'))


Tasa de fraude por TIPO DE PÓLIZA BASE:

            n_total  n_fraudes  tasa_fraude  tasa_fraude_%
BasePolicy                                                
All Perils     4449        452       0.1016          10.16
Collision      5962        435       0.0730           7.30
Liability      5009         36       0.0072           0.72

Tasa de fraude por RESPONSABLE (Fault):

               n_total  n_fraudes  tasa_fraude  tasa_fraude_%
Fault                                                        
Policy Holder    11230        886       0.0789           7.89
Third Party       4190         37       0.0088           0.88

Tasa de fraude por ZONA DE ACCIDENTE:

              n_total  n_fraudes  tasa_fraude  tasa_fraude_%
AccidentArea                                                
Rural            1598        133       0.0832           8.32
Urban           13822        790       0.0572           5.72


In [83]:
# 6.3 Correlaciones (variables numéricas)

corr = df_clean.select_dtypes(include=np.number).corr()
fig = px.imshow(
    corr.round(2), text_auto=True, aspect='auto',
    color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
    title='Matriz de correlación de variables numéricas y ordinales'
)
fig.update_layout(width=900, height=750)
fig.show()



**Justificacion:** La matriz revela una estructura de datos saludable con baja multicolinealidad,las asociaciones más significativas son esperables dentro del negocio: Age y AgeOfVehicle (0.50), Days_Policy_Accident y Days_Policy_Claim (0.53), y AddressChange_Claim con NumberOfCars (0.47), es importante notar que la variable FraudFound_P presenta correlaciones cercanas a cero con el resto de las variables individuales, esto confirma que el fraude no es un fenómeno lineal simple.

## 7. Análisis Exploratorio de Datos (EDA) con Plotly

In [84]:
# 7.1 Distribución de Edad por clase
fig = px.histogram(
    df_clean, x='Age', color='FraudFound_P', barmode='overlay',
    nbins=40, opacity=0.7,
    color_discrete_map={0: '#2E86AB', 1: '#E63946'},
    title='Distribución de edad por clase (Fraude vs Legítimo)',
    labels={'FraudFound_P': 'Fraude'}
)
fig.update_layout(bargap=0.05)
fig.show()


**Justificación:** La distribución de edad es similar entre fraudes y casos legítimos, con concentración entre los 25 y 45 años. La edad por sí sola **no es un discriminador fuerte**, pero podría aportar señal en combinación con otras variables (interacciones).

In [85]:
# 7.2 Boxplot de Age por Fraude y Sexo
fig = px.box(
    df_clean, x='Sex', y='Age', color='FraudFound_P',
    color_discrete_map={0: '#2E86AB', 1: '#E63946'},
    title='Edad por sexo y clase'
)
fig.show()


**Justificación:** Se observa que tanto en hombres como en mujeres las medianas y los rangos intercuartílicos son muy similares entre las clases, esto indica que no existen diferencias significativas de edad según el sexo para los casos de fraude; en conclusión, el fraude no parece depender fuertemente del perfil demográfico  (sexo y edad), ya que las distribuciones son bastante comparables entre ambas clases.

In [86]:
# 7.3 Tasa de fraude por BasePolicy y Fault (heatmap segmentado)
pivot = df_clean.pivot_table(
    index='BasePolicy', columns='Fault',
    values='FraudFound_P', aggfunc='mean'
).round(4) * 100

fig = px.imshow(
    pivot, text_auto='.2f', aspect='auto',
    color_continuous_scale='Reds',
    title='Tasa de fraude (%) por BasePolicy × Fault'
)
fig.update_layout(width=700, height=400)
fig.show()


**Justificación:** Las pólizas tipo **"All Perils"** y **"Collision"** donde el responsable es el **propio asegurado (Policy Holder)** muestran la **tasa de fraude más alta** (10–16%). En cambio, las pólizas **"Liability"** son prácticamente libres de fraude. Esto tiene sentido: las pólizas que cubren al propio asegurado tienen mayor incentivo a la simulación. **`BasePolicy` y `Fault` son variables altamente predictivas.**

In [87]:
# 7.4 Tasa de fraude por mes del accidente 
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df_clean.groupby('Month')['FraudFound_P'].agg(['count','mean']).reset_index()
monthly['Month'] = pd.Categorical(monthly['Month'], categories=month_order, ordered=True)
monthly = monthly.sort_values('Month')
monthly['fraud_rate_%'] = (monthly['mean']*100).round(2)

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(x=monthly['Month'], y=monthly['count'],
                     name='Total reclamaciones', marker_color='#2E86AB'), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly['Month'], y=monthly['fraud_rate_%'],
                         name='Tasa fraude (%)', mode='lines+markers',
                         marker=dict(size=10, color='#E63946'), line=dict(width=3)), secondary_y=True)
fig.update_layout(title='Volumen de reclamaciones y tasa de fraude por mes', height=450)
fig.update_yaxes(title='Total reclamaciones', secondary_y=False)
fig.update_yaxes(title='Tasa fraude (%)', secondary_y=True)
fig.show()


**Justificación:** El volumen de reclamaciones es relativamente estable a lo largo del año, con ligeros picos en marzo y mayo. La **tasa de fraude oscila entre 5% y 7%** sin un patrón estacional claro, sugiriendo que el fraude **no depende fuertemente del mes calendario**.

In [88]:
# 7.5 Reclamaciones previas vs fraude 
fig = px.histogram(
    df_clean, x='PastNumberOfClaims', color='FraudFound_P',
    barmode='group', histnorm='percent',
    color_discrete_map={0: '#2E86AB', 1: '#E63946'},
    title='Reclamaciones previas (codificadas 0–3) por clase',
    labels={'PastNumberOfClaims': '0=ninguna, 1=una, 2=2-4, 3=más de 4'}
)
fig.show()


**Justificación:** La gráfica confirma que el fraude es mucho más frecuente en personas que ya tienen un historial de varios reclamos, no es que tener un accidente sea malo, pero tener accidentes de forma repetida es una "bandera roja" que obliga a la empresa a investigar más a fondo.

In [89]:
# 7.6 Tasa de fraude por marca del vehículo
make_stats = (df_clean.groupby('Make')['FraudFound_P']
              .agg(['count','mean']).reset_index()
              .query('count >= 50')  # filtrar marcas con muestra suficiente
              .sort_values('mean', ascending=False))
make_stats['fraud_rate_%'] = (make_stats['mean']*100).round(2)

fig = px.bar(
    make_stats, x='Make', y='fraud_rate_%', color='count',
    color_continuous_scale='Blues',
    title='Tasa de fraude (%) por marca de vehículo (n ≥ 50)',
    labels={'count': 'Número de reclamaciones'}
)
fig.add_hline(y=df_clean['FraudFound_P'].mean()*100, line_dash='dash',
              line_color='red', annotation_text='Tasa promedio (5.99%)')
fig.show()


**Justificación:** Aunque marcas como Accura, Saturn y Saab presentan las tasas de fraude más altas, estas marcas tienen un volumen de reclamaciones muy bajo, en contraste, marcas con alto volumen como Pontiac y Toyota se mantienen cerca del promedio del 6% lo que indica que la marca del auto por sí sola no es un predictor definitivo; es probable que el fraude dependa más del perfil del conductor que del logo del vehículo.

## 8 Preprocesamiento con Scikit-Learn Pipelines

- **Variables numéricas / ordinales** → `StandardScaler` (necesario para Regresión Logística).
- **Variables categóricas nominales** → `OneHotEncoder` (sin orden natural).


In [90]:
# 8.1 División de variables y separación X / y
TARGET = 'FraudFound_P'

# Identificación de tipos
nominal_cols = ['Month', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed',
                'MonthClaimed', 'Sex', 'MaritalStatus', 'Fault', 'PolicyType',
                'VehicleCategory', 'PoliceReportFiled', 'WitnessPresent',
                'AgentType', 'BasePolicy']

numeric_cols = [c for c in df_clean.columns
                if c not in nominal_cols + [TARGET]]


X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

# 8.2 Split estratificado (preserva la proporción de la clase minoritaria) 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"\n Train: {X_train.shape} | Test: {X_test.shape}")
print(f" Tasa fraude train: {y_train.mean():.4f} | test: {y_test.mean():.4f}")



 Train: (12336, 29) | Test: (3084, 29)
 Tasa fraude train: 0.0598 | test: 0.0600


In [91]:
# 8.3 Construcción del ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_cols)
    ],
    remainder='drop'
)

# Probarlo
X_train_pp = preprocessor.fit_transform(X_train)
print(f"({X_train_pp.shape[1]} features luego del OneHotEncoding)")


(102 features luego del OneHotEncoding)


## 9. Modelos estadísticos y de Machine Learning

### 9.1 Modelos seleccionados y justificación

| # | Modelo | Justificación | Trade-offs |
|---|--------|---------------|-----------|
| **M1** | **Regresión Logística** + `class_weight='balanced'` | Baseline interpretable. Permite analizar coeficientes, log-odds y odds ratios. | Asume linealidad en log-odds. |
| **M2** | **Random Forest** + `class_weight='balanced'` | Captura interacciones no lineales. Robusto a outliers. Provee importancia de variables nativa. | Menos interpretable a nivel individual. |
| **M3** | **XGBoost** con `scale_pos_weight` | Estado del arte en datos tabulares. Manejo eficiente del desbalance. | Más hiperparámetros, mayor costo computacional. |
| **M4** | **Logística + SMOTE** | Compara el enfoque de **reponderación** (M1) vs **sobremuestreo sintético** (M4). | SMOTE puede generar puntos sintéticos poco realistas en variables one-hot. |

### 9.2 Estrategia de evaluación

Dado el desbalance, la métrica clave es **PR-AUC** (más robusta que ROC-AUC), seguida de **Recall** (no perder fraudes) y **F1-score** (balance recall-precisión).

In [92]:
# 9.3 Definición de los 4 pipelines de modelo

# M1: Regresión Logística
pipe_logreg = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=2000,
                               random_state=RANDOM_STATE, solver='lbfgs'))
])

# M2: Random Forest
pipe_rf = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=5,
        class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE))
])

# M3: XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight para XGBoost: {scale_pos_weight:.2f}")

pipe_xgb = Pipeline([
    ('prep', preprocessor),
    ('clf', XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.05,
        scale_pos_weight=scale_pos_weight, eval_metric='logloss',
        n_jobs=-1, random_state=RANDOM_STATE))
])

# M4: Logística + SMOTE
pipe_smote = ImbPipeline([
    ('prep', preprocessor),
    ('smote', SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
    ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, solver='lbfgs'))
])

models = {
    'Logistic Regression': pipe_logreg,
    'Random Forest': pipe_rf,
    'XGBoost': pipe_xgb,
    'Logistic + SMOTE': pipe_smote
}
print("4 pipelines configurados.")


scale_pos_weight para XGBoost: 15.72
4 pipelines configurados.


In [93]:
# 9.4 Entrenamiento de los 4 modelos 
import time
trained = {}
for name, pipe in models.items():
    t0 = time.time()
    pipe.fit(X_train, y_train)
    trained[name] = pipe
    print(f"{name:25s} entrenado en {time.time()-t0:5.1f}s")


Logistic Regression       entrenado en   0.6s


Random Forest             entrenado en   2.4s
XGBoost                   entrenado en   1.0s
Logistic + SMOTE          entrenado en   1.2s


## 10. Evaluación de los modelos

In [94]:
# 10.1 Cálculo de métricas en el set de prueba
def evaluate(model, X_te, y_te):
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    return {
        'Accuracy':   accuracy_score(y_te, y_pred),
        'Precision':  precision_score(y_te, y_pred, zero_division=0),
        'Recall':     recall_score(y_te, y_pred),
        'F1':         f1_score(y_te, y_pred),
        'ROC-AUC':    roc_auc_score(y_te, y_proba),
        'PR-AUC':     average_precision_score(y_te, y_proba)
    }

results = {name: evaluate(m, X_test, y_test) for name, m in trained.items()}
results_df = pd.DataFrame(results).T.round(4)
print("Métricas en el set de prueba\n")
print(results_df)


Métricas en el set de prueba

                     Accuracy  Precision  Recall      F1  ROC-AUC  PR-AUC
Logistic Regression    0.6440     0.1316  0.8811  0.2289   0.8038  0.1696
Random Forest          0.7218     0.1462  0.7514  0.2447   0.8124  0.1810
XGBoost                0.8061     0.1575  0.5135  0.2411   0.7984  0.2071
Logistic + SMOTE       0.6621     0.1316  0.8270  0.2270   0.8014  0.1663


In [95]:
# 10.2 Visualización comparativa de métricas 
results_long = results_df.reset_index().melt(
    id_vars='index', var_name='Métrica', value_name='Valor'
).rename(columns={'index': 'Modelo'})

fig = px.bar(
    results_long, x='Métrica', y='Valor', color='Modelo',
    barmode='group', text_auto='.3f',
    title='Comparación de métricas entre modelos',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(height=500, yaxis_range=[0, 1])
fig.show()


In [96]:
# 10.3 Matrices de confusión
fig = make_subplots(rows=2, cols=2, subplot_titles=list(trained.keys()))
positions = [(1,1),(1,2),(2,1),(2,2)]
for (name, model), (r, c) in zip(trained.items(), positions):
    cm = confusion_matrix(y_test, model.predict(X_test))
    fig.add_trace(go.Heatmap(
        z=cm, x=['Pred 0','Pred 1'], y=['Real 0','Real 1'],
        text=cm, texttemplate='%{text}', textfont={'size':14},
        colorscale='Blues', showscale=False
    ), row=r, col=c)
fig.update_layout(height=700, title_text='Matrices de confusión por modelo')
fig.show()


In [97]:
# 10.4 Curvas ROC y Precision-Recall
fig = make_subplots(rows=1, cols=2, subplot_titles=('Curva ROC', 'Curva Precision-Recall'))

for name, model in trained.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                             name=f'{name} (AUC={auc:.3f})', legendgroup=name),
                  row=1, col=1)
    # PR
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    fig.add_trace(go.Scatter(x=rec, y=prec, mode='lines',
                             name=f'{name} (AP={ap:.3f})', legendgroup=name,
                             showlegend=False),
                  row=1, col=2)

# Línea diagonal ROC
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
                         line=dict(dash='dash', color='gray'),
                         showlegend=False), row=1, col=1)

fig.update_xaxes(title_text='False Positive Rate', row=1, col=1)
fig.update_yaxes(title_text='True Positive Rate',  row=1, col=1)
fig.update_xaxes(title_text='Recall',    row=1, col=2)
fig.update_yaxes(title_text='Precision', row=1, col=2)
fig.update_layout(height=500, title_text='Curvas ROC y PR — Comparación de modelos')
fig.show()


In [98]:
# 10.5 Selección del mejor modelo 
best_model_name = results_df['PR-AUC'].idxmax()
best_model = trained[best_model_name]
print(f" Mejor modelo según PR-AUC: {best_model_name}")
print(f" PR-AUC = {results_df.loc[best_model_name, 'PR-AUC']:.4f}")
print(f" Recall = {results_df.loc[best_model_name, 'Recall']:.4f}")
print(f" F1     = {results_df.loc[best_model_name, 'F1']:.4f}")


 Mejor modelo según PR-AUC: XGBoost
 PR-AUC = 0.2071
 Recall = 0.5135
 F1     = 0.2411


## 11. Interpretabilidad

1. **Coeficientes y odds ratios** de la Regresión Logística.
2. **Importancia de variables** nativa del Random Forest.
3. **Permutation importance** (independiente del modelo, más robusta).
4. **SHAP values** sobre el modelo XGBoost (explicaciones locales y globales).

In [99]:
# 11.1 Coeficientes y odds ratios — Regresión Logística
def get_feature_names(pipe):
    pre = pipe.named_steps['prep']
    num_names = numeric_cols
    cat_names = pre.named_transformers_['cat'].get_feature_names_out(nominal_cols).tolist()
    return num_names + cat_names

feat_names = get_feature_names(pipe_logreg)
coefs = pipe_logreg.named_steps['clf'].coef_[0]

logreg_df = pd.DataFrame({
    'feature': feat_names,
    'coef': coefs,
    'odds_ratio': np.exp(coefs)
}).sort_values('coef', key=abs, ascending=False)

print("Top 15 coeficientes (en valor absoluto):\n")
print(logreg_df.head(15).round(4).to_string(index=False))


Top 15 coeficientes (en valor absoluto):

                        feature    coef  odds_ratio
              Fault_Third Party -1.6119      0.1995
  PolicyType_Sport - All Perils -1.2346      0.2909
            Fault_Policy Holder  1.1841      3.2677
           BasePolicy_Liability -1.1788      0.3077
   PolicyType_Sport - Collision  1.1778      3.2474
                     Make_Dodge -1.1081      0.3302
                        Make_VW -1.0055      0.3659
                      Month_Jul -0.7780      0.4593
             AgentType_Internal -0.7662      0.4648
 PolicyType_Utility - Liability -0.7569      0.4691
PolicyType_Utility - All Perils  0.6683      1.9508
  PolicyType_Sedan - All Perils  0.6611      1.9369
           BasePolicy_Collision  0.6563      1.9276
                    Make_Accura  0.6487      1.9131
               MonthClaimed_Aug  0.6225      1.8637


In [100]:
# 11.2 Visualización de coeficientes top
top15 = logreg_df.head(15).sort_values('coef')
fig = px.bar(
    top15, x='coef', y='feature', orientation='h',
    color='coef', color_continuous_scale='RdBu_r', color_continuous_midpoint=0,
    title='Top 15 coeficientes — Regresión Logística (en escala estandarizada)',
    labels={'coef':'Coeficiente (log-odds)','feature':'Variable'}
)
fig.update_layout(height=550)
fig.show()


**Justificación de odds ratios:** El factor de riesgo más crítico para el fraude es la responsabilidad del siniestro, las variables PolicyType_Sport - Collision y Fault_Policy Holder presentan los coeficientes positivos más altos, lo que indica que cuando el asegurado es el responsable del choque las probabilidades de fraude se disparan. Por el contrario, cuando la responsabilidad recae en un tercero (Fault_Third Party), el riesgo de fraude disminuye drásticamente.

In [101]:
# 11.3 Importancia de variables — Random Forest
rf_importances = pipe_rf.named_steps['clf'].feature_importances_
rf_df = pd.DataFrame({'feature': feat_names, 'importance': rf_importances})
rf_df = rf_df.sort_values('importance', ascending=False).head(15)

fig = px.bar(
    rf_df.sort_values('importance'), x='importance', y='feature',
    orientation='h', color='importance', color_continuous_scale='Viridis',
    title='Top 15 importancia de variables — Random Forest'
)
fig.update_layout(height=550)
fig.show()


**Justificación:** se evidencia que el factor determinante para detectar el fraude es la responsabilidad del siniestro (Fault), las dos variables con mayor importancia son si la culpa fue del asegurado (Fault_Policy Holder) o de un tercero (Fault_Third Party), para el modelo es mas importante quien tuvo la culpa que otras variables como la edad o el precio del vehículo.

In [102]:
# 11.4 Permutation Importance sobre el mejor modelo
print(f"Calculando permutation importance sobre {best_model_name} (puede tardar 1-2 min)...")
perm = permutation_importance(
    best_model, X_test, y_test, n_repeats=5,
    random_state=RANDOM_STATE, n_jobs=-1, scoring='average_precision'
)

# Mapeamos a nombres originales (no a los expandidos por OneHot)
perm_df = pd.DataFrame({
    'feature': X_test.columns,
    'mean_importance': perm.importances_mean,
    'std': perm.importances_std
}).sort_values('mean_importance', ascending=False).head(15)

fig = px.bar(
    perm_df.sort_values('mean_importance'), x='mean_importance', y='feature',
    orientation='h', error_x='std',
    color='mean_importance', color_continuous_scale='Plasma',
    title=f'Top 15 Permutation Importance — {best_model_name}'
)
fig.update_layout(height=550)
fig.show()


Calculando permutation importance sobre XGBoost (puede tardar 1-2 min)...


**Justificación:** la Grafica confirma que  las variables Fault y BasePolicy  son los pilares fundamentales del modelo XGBoost, al ser las variables con mayor mean_importance, el modelo depende críticamente de ellas para predecir el fraude, ademas es importante resaltar como la variable AddressChange_Claim tiene un peso relevante, sugiriendo que cambios recientes en el perfil del cliente son señales de alerta importantes para este algoritmo.

In [103]:
# === 11.5 SHAP values sobre XGBoost (Versión Universal) ===
print("Calculando SHAP values con método de permutación (evitando error de base_score)...")

# 1. Preparar los datos
X_test_pp = pipe_xgb.named_steps['prep'].transform(X_test)
# Usamos una muestra de 100 para que sea rápido, ya que este método es más intensivo
X_test_sample = X_test_pp[:100] 

# 2. Definir la función de predicción (usamos la probabilidad de la clase positiva)
# Esto envuelve al modelo para que SHAP lo vea como una función matemática simple
def model_predict(data):
    # Aseguramos que los datos sean el formato que espera el clasificador
    return pipe_xgb.named_steps['clf'].predict_proba(data)[:, 1]

# 3. Crear el explicador genérico
# Usamos masker para que SHAP sepa cómo "ocultar" características
masker = shap.maskers.Independent(X_test_pp, max_samples=100)
explainer = shap.Explainer(model_predict, masker)

# 4. Calcular SHAP values
# Nota: Aquí no usamos .shap_values(), sino que llamamos al objeto explainer directamente
shap_values = explainer(X_test_sample)

# 5. Resumen global
# shap_values.values contiene la matriz de importancia
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)

# 6. Crear DataFrame para graficar
shap_df = pd.DataFrame({
    'feature': feat_names,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False).head(15)

# 7. Gráfico con Plotly
fig = px.bar(
    shap_df.sort_values('mean_abs_shap'), 
    x='mean_abs_shap', 
    y='feature',
    orientation='h', 
    color='mean_abs_shap', 
    color_continuous_scale='Sunset',
    title='Top 15 importancia SHAP (Permutation) — XGBoost'
)
fig.update_layout(height=550)
fig.show()

Calculando SHAP values con método de permutación (evitando error de base_score)...


PermutationExplainer explainer: 101it [00:10,  1.77s/it]                        


**Justificación:** El análisis de valores SHAP confirma de manera contundente que la culpabilidad del asegurado (Fault_Policy Holder) es el factor con mayor peso predictivo en el modelo XGBoost, superando significativamente al resto de variables. La segunda variable más relevante es el tipo de póliza base (BasePolicy_Liability), la gran diferencia en la magnitud de los valores SHAP entre estas dos y las demás sugiere que el modelo toma sus decisiones basándose principalmente en la responsabilidad legal y el tipo de contrato.

## 12. Conclusiones

1. **Fuerte desbalance (≈6% fraude):** las métricas estándar (accuracy) son engañosas. **PR-AUC y Recall** son las métricas críticas para la aseguradora.
2. **Variables más predictivas:** `Fault`, `BasePolicy`, `PastNumberOfClaims`, `AddressChange_Claim`, `AgeOfPolicyHolder` y `VehiclePrice` aparecen consistentemente en los tres rankings de importancia.
3. **Patrón de negocio confirmado:** las pólizas tipo *All Perils* y *Collision* donde el responsable es el propio asegurado tienen **tasa de fraude 2-3× la media**.
4. **Ausencia de patrón temporal:** ni el mes ni el día de la semana aportan señal significativa.
5. **Modelos no lineales superan al lineal:** Random Forest y XGBoost capturan interacciones que la Logística (con o sin SMOTE) no puede.

## 13. Aplicación Streamlit

In [104]:
# 13.1 Guardado de artefactos para la app
import joblib
import os

# Guardar el mejor modelo (compatibilidad)
joblib.dump(best_model, '../models/best_fraud_model.pkl')

# Guardar TODOS los modelos para que la app permita compararlos
os.makedirs('models', exist_ok=True)
for name, model in trained.items():
    safe_name = name.replace(' ', '_').replace('+', 'plus')
    joblib.dump(model, f'models/{safe_name}.pkl')
    print(f"  Guardado: models/{safe_name}.pkl")

# Guardar metadata
joblib.dump({
    'numeric_cols': numeric_cols,
    'nominal_cols': nominal_cols,
    'ordinal_mappings': ordinal_mappings,
    'best_model_name': best_model_name,
    'feature_names': feat_names,
    'all_model_names': list(trained.keys()),
    'results': results_df.to_dict()
}, '../models/model_metadata.pkl')

# Dataset limpio
df_clean.to_csv('fraud_clean.csv', index=False)

print(f"\nMejor modelo según PR-AUC: '{best_model_name}'")
print(f"Metadata guardada en model_metadata.pkl")
print(f"Dataset limpio guardado en fraud_clean.csv")
print(f"\n4 modelos disponibles para la app:")
for n in trained.keys():
    print(f"  • {n}")


  Guardado: models/Logistic_Regression.pkl
  Guardado: models/Random_Forest.pkl
  Guardado: models/XGBoost.pkl
  Guardado: models/Logistic_plus_SMOTE.pkl

Mejor modelo según PR-AUC: 'XGBoost'
Metadata guardada en model_metadata.pkl
Dataset limpio guardado en fraud_clean.csv

4 modelos disponibles para la app:
  • Logistic Regression
  • Random Forest
  • XGBoost
  • Logistic + SMOTE


In [107]:
# === 13.2 Generación del archivo app.py ===
# Se escribe línea por línea para evitar conflictos de escape.

app_lines = [
    '"""',
    '🔍 App de Detección de Fraude en Seguros Vehiculares',
    'Construida con Streamlit + Plotly + Scikit-Learn / XGBoost',
    'Ejecutar:  streamlit run app.py',
    '"""',
    'import os',
    'import streamlit as st',
    'import pandas as pd',
    'import numpy as np',
    'import joblib',
    'import plotly.express as px',
    'import plotly.graph_objects as go',
    'from plotly.subplots import make_subplots',
    'from sklearn.metrics import (confusion_matrix, classification_report,',
    '                             roc_auc_score, average_precision_score, roc_curve,',
    '                             precision_recall_curve, accuracy_score,',
    '                             precision_score, recall_score, f1_score)',
    'from sklearn.model_selection import train_test_split',
    '',
    '# --------------------------------------------------------------',
    '# Configuración de la página',
    '# --------------------------------------------------------------',
    'st.set_page_config(page_title="Fraud Detection", page_icon="🔍", layout="wide")',
    'st.title("🔍 Detección de Fraude en Seguros Vehiculares")',
    'st.markdown(',
    '    "Aplicación académica — Universidad de Medellín — Proyecto 1 | "',
    '    "Profesor: David Palacio Jimenez | "',
    '    "Estudiantes: Andres Felipe Londoño Ocampo · Jhonatan Caro Atehortúa · "',
    '    "Paulina Perez Ramirez · Victor Manuel Galeano Alvarez"',
    ')',
    '',
    '# --------------------------------------------------------------',
    '# Mapeos ordinales',
    '# --------------------------------------------------------------',
    'ORDINAL_MAPPINGS = {',
    '    \'VehiclePrice\': {',
    '        \'less than 20000\': 0, \'20000 to 29000\': 1, \'30000 to 39000\': 2,',
    '        \'40000 to 59000\': 3, \'60000 to 69000\': 4, \'more than 69000\': 5',
    '    },',
    '    \'Days_Policy_Accident\': {',
    '        \'none\': 0, \'1 to 7\': 1, \'8 to 15\': 2, \'15 to 30\': 3, \'more than 30\': 4',
    '    },',
    '    \'Days_Policy_Claim\': {',
    '        \'none\': 0, \'8 to 15\': 1, \'15 to 30\': 2, \'more than 30\': 3',
    '    },',
    '    \'PastNumberOfClaims\': {',
    '        \'none\': 0, \'1\': 1, \'2 to 4\': 2, \'more than 4\': 3',
    '    },',
    '    \'AgeOfVehicle\': {',
    '        \'new\': 0, \'2 years\': 1, \'3 years\': 2, \'4 years\': 3,',
    '        \'5 years\': 4, \'6 years\': 5, \'7 years\': 6, \'more than 7\': 7',
    '    },',
    '    \'NumberOfSuppliments\': {',
    '        \'none\': 0, \'1 to 2\': 1, \'3 to 5\': 2, \'more than 5\': 3',
    '    },',
    '    \'AddressChange_Claim\': {',
    '        \'no change\': 0, \'under 6 months\': 1, \'1 year\': 2,',
    '        \'2 to 3 years\': 3, \'4 to 8 years\': 4',
    '    },',
    '    \'NumberOfCars\': {',
    '        \'1 vehicle\': 1, \'2 vehicles\': 2, \'3 to 4\': 3,',
    '        \'5 to 8\': 4, \'more than 8\': 5',
    '    }',
    '}',
    '',
    'FILTROS_EDA = {',
    '    \'Month\':              \'Mes del accidente\',',
    '    \'DayOfWeek\':          \'Día de la semana del accidente\',',
    '    \'MonthClaimed\':       \'Mes de la reclamación\',',
    '    \'DayOfWeekClaimed\':   \'Día semana reclamación\',',
    '    \'Make\':               \'Marca del vehículo\',',
    '    \'AccidentArea\':       \'Zona del accidente\',',
    '    \'Sex\':                \'Sexo del asegurado\',',
    '    \'MaritalStatus\':      \'Estado civil\',',
    '    \'Fault\':              \'Responsable del siniestro\',',
    '    \'PolicyType\':         \'Tipo de póliza\',',
    '    \'VehicleCategory\':    \'Categoría del vehículo\',',
    '    \'PoliceReportFiled\':  \'Reporte policial\',',
    '    \'WitnessPresent\':     \'Testigos presentes\',',
    '    \'AgentType\':          \'Tipo de agente\',',
    '    \'BasePolicy\':         \'Póliza base\',',
    '}',
    '',
    '# --------------------------------------------------------------',
    '# Carga cacheada',
    '# --------------------------------------------------------------',
   '@st.cache_resource',
    'def load_artifacts():',
    '    import os',
    '    # 1. Buscar la metadata (intenta en raíz y luego en models/)',
    '    meta_path = "model_metadata.pkl" if os.path.exists("model_metadata.pkl") else "models/model_metadata.pkl"',
    '    if not os.path.exists(meta_path):',
    '        raise FileNotFoundError(f"No se encontró model_metadata.pkl en la raíz ni en models/")',
    '    ',
    '    meta = joblib.load(meta_path)',
    '    models_dict = {}',
    '    ',
    '    # 2. Definir dónde buscar los modelos individuales',
    '    folder = "models" if os.path.isdir("models") else "."',
    '    ',
    '    for name in meta.get(\'all_model_names\', []):',
    '        safe_name = name.replace(\' \', \'_\').replace(\'+\', \'plus\')',
    '        path = os.path.join(folder, f"{safe_name}.pkl")',
    '        if os.path.exists(path):',
    '            models_dict[name] = joblib.load(path)',
    '    ',
    '    # 3. Fallback: Si no encontró los 4, cargar al menos el "mejor modelo"',
    '    if not models_dict:',
    '        best_path = "best_fraud_model.pkl" if os.path.exists("best_fraud_model.pkl") else "models/best_fraud_model.pkl"',
    '        if os.path.exists(best_path):',
    '            models_dict[meta[\'best_model_name\']] = joblib.load(best_path)',
    '    ',
    '    return models_dict, meta',
    '',
    '@st.cache_data',
    'def load_data():',
    '    return pd.read_csv("fraud_clean.csv")',
    '',
    '@st.cache_data',
    'def compute_summary_metrics(_models_dict):',
    '    """Calcula métricas de los modelos sobre el set de prueba."""',
    '    df_local = pd.read_csv("fraud_clean.csv")',
    '    X = df_local.drop(columns=[\'FraudFound_P\'])',
    '    y = df_local[\'FraudFound_P\']',
    '    _, X_te, _, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)',
    '    rows = []',
    '    for name, m in _models_dict.items():',
    '        y_pred  = m.predict(X_te)',
    '        y_proba = m.predict_proba(X_te)[:, 1]',
    '        rows.append({',
    '            \'Modelo\':    name,',
    '            \'Accuracy\':  accuracy_score(y_te, y_pred),',
    '            \'Precision\': precision_score(y_te, y_pred, zero_division=0),',
    '            \'Recall\':    recall_score(y_te, y_pred),',
    '            \'F1\':        f1_score(y_te, y_pred),',
    '            \'ROC-AUC\':   roc_auc_score(y_te, y_proba),',
    '            \'PR-AUC\':    average_precision_score(y_te, y_proba),',
    '        })',
    '    return pd.DataFrame(rows)',
    '',
    'models_dict, meta = load_artifacts()',
    'df = load_data()',
    'all_model_names = list(models_dict.keys())',
    'best_model_name = meta[\'best_model_name\']',
    '',
    '# --------------------------------------------------------------',
    '# Sidebar',
    '# --------------------------------------------------------------',
    'st.sidebar.title("📋 Menú")',
    'page = st.sidebar.radio("Navegar a:", [',
    '    "🏠 Inicio",',
    '    "📊 EDA Interactivo",',
    '    "🎯 Predicción",',
    '    "📈 Desempeño del modelo"',
    '])',
    'st.sidebar.markdown("---")',
    '',
    'st.sidebar.subheader("⚙️ Modelo activo")',
    'selected_model_name = st.sidebar.selectbox(',
    '    "Selecciona un modelo:",',
    '    all_model_names,',
    '    index=all_model_names.index(best_model_name) if best_model_name in all_model_names else 0',
    ')',
    'selected_model = models_dict[selected_model_name]',
    'if selected_model_name == best_model_name:',
    '    st.sidebar.success(f"⭐ Mejor modelo: {selected_model_name}")',
    'else:',
    '    st.sidebar.info(f"Modelo: {selected_model_name}")',
    '',
    '# --------------------------------------------------------------',
    '# Página 1 – Inicio (RESUMEN EJECUTIVO)',
    '# --------------------------------------------------------------',
    'if page == "🏠 Inicio":',
    '    # Banner principal',
    '    st.header("📋 Resumen Ejecutivo del Proyecto")',
    '',
    '    # ============ KPIs principales ============',
    '    metrics_df = compute_summary_metrics(models_dict)',
    '    best_pr_auc = metrics_df.loc[metrics_df[\'Modelo\'] == best_model_name, \'PR-AUC\'].values[0] if best_model_name in metrics_df[\'Modelo\'].values else metrics_df[\'PR-AUC\'].max()',
    '    best_roc_auc = metrics_df.loc[metrics_df[\'Modelo\'] == best_model_name, \'ROC-AUC\'].values[0] if best_model_name in metrics_df[\'Modelo\'].values else metrics_df[\'ROC-AUC\'].max()',
    '    best_recall = metrics_df.loc[metrics_df[\'Modelo\'] == best_model_name, \'Recall\'].values[0] if best_model_name in metrics_df[\'Modelo\'].values else metrics_df[\'Recall\'].max()',
    '',
    '    k1, k2, k3, k4, k5, k6 = st.columns(6)',
    '    k1.metric("Reclamaciones", f"{len(df):,}")',
    '    k2.metric("Tasa de fraude", f"{df[\'FraudFound_P\'].mean()*100:.2f}%")',
    '    k3.metric("Variables", df.shape[1] - 1)',
    '    k4.metric("Modelos", len(all_model_names))',
    '    k5.metric("Mejor PR-AUC", f"{best_pr_auc:.3f}")',
    '    k6.metric("Mejor Recall", f"{best_recall:.3f}")',
    '',
    '    st.markdown("---")',
    '',
    '    # ============ CONTEXTO Y PROBLEMÁTICA ============',
    '    col_a, col_b = st.columns(2)',
    '',
    '    with col_a:',
    '        st.subheader("🎯 Resumen del Proyecto")',
    '        st.markdown("""',
    '        Análisis y modelado predictivo para la **detección automática de fraude**',
    '        en reclamaciones de seguros vehiculares, utilizando el dataset',
    '        *Vehicle Insurance Claim Fraud Detection* (Oracle/Kaggle) con',
    '        15,420 registros del período 1994-1996.',
    '',
    '        Se aplicaron técnicas de **EDA**, **preprocesamiento con',
    '        Pipelines de Scikit-Learn** y se entrenaron **4 modelos de clasificación**',
    '        comparándolos sobre métricas adecuadas para el desbalance de clases.',
    '        """)',
    '',
    '        st.subheader("⚠️ Problemática")',
    '        st.markdown("""',
    '        El fraude en seguros vehiculares representa **pérdidas multimillonarias**',
    '        para la industria aseguradora a nivel global y encarece indirectamente',
    '        las primas de los asegurados honestos.',
    '',
    '        La detección manual es **lenta, costosa y propensa a errores humanos**,',
    '        lo que hace inviable revisar el 100% de las reclamaciones que recibe',
    '        una compañía aseguradora.',
    '        """)',
    '',
    '    with col_b:',
    '        st.subheader("💼 Contexto de Negocio")',
    '        st.markdown("""',
    '        La aseguradora necesita un **sistema de soporte a la decisión** que asigne',
    '        una *probabilidad de fraude* a cada reclamación nueva, permitiendo a los',
    '        analistas **priorizar la revisión** de los casos de mayor riesgo.',
    '',
    '        Un modelo predictivo bien calibrado:',
    '        - **Reduce costos operativos** (menos casos a revisar manualmente)',
    '        - **Aumenta la tasa de detección** sin generar fricción con clientes legítimos',
    '        - **Genera trazabilidad** para auditorías y cumplimiento regulatorio',
    '        """)',
    '',
    '        # Gráfico de torta del desbalance',
    '        fraud_counts = df[\'FraudFound_P\'].value_counts()',
    '        fig_pie = px.pie(',
    '            values=fraud_counts.values,',
    '            names=[\'Legítimo\', \'Fraude\'],',
    '            title=\'Distribución de la variable objetivo\',',
    '            color_discrete_sequence=[\'#2E86AB\', \'#E63946\'],',
    '            hole=0.5',
    '        )',
    '        fig_pie.update_traces(textinfo=\'percent+label+value\')',
    '        fig_pie.update_layout(',
    '            height=350,',
    '            margin=dict(t=80, b=20, l=20, r=20),',
    '            title_y=0.98,',
    '            legend=dict(orientation=\'v\', yanchor=\'middle\', y=0.5, xanchor=\'left\', x=1.05)',
    '        )',
    '        st.plotly_chart(fig_pie, use_container_width=True)',
    '',
    '    st.markdown("---")',
    '',
    '    # ============ METODOLOGÍA ============',
    '    st.subheader("🔬 Metodología aplicada")',
    '    met_cols = st.columns(5)',
    '    with met_cols[0]:',
    '        st.markdown("**1️⃣ Comprensión**")',
    '        st.caption("Definición del problema, exploración inicial del dataset y diccionario de variables.")',
    '    with met_cols[1]:',
    '        st.markdown("**2️⃣ Preprocesamiento**")',
    '        st.caption("Tratamiento de datos faltantes, codificación ordinal y exclusión de variables (Year, AgeOfPolicyHolder, PolicyNumber).")',
    '    with met_cols[2]:',
    '        st.markdown("**3️⃣ EDA**")',
    '        st.caption("Análisis univariado y bivariado, correlaciones, tasas de fraude por categoría.")',
    '    with met_cols[3]:',
    '        st.markdown("**4️⃣ Modelado**")',
    '        st.caption("Pipelines con StandardScaler, OneHotEncoder y 4 modelos comparados.")',
    '    with met_cols[4]:',
    '        st.markdown("**5️⃣ Interpretación**")',
    '        st.caption("Coeficientes, importancias, permutation importance y SHAP values.")',
    '',
    '',
    '    st.markdown("---")',
    '',
    '    # ============ MEJOR MODELO ============',
    '    st.subheader("🏆 Mejor modelo seleccionado")',
    '',
    '    # Métricas completas del mejor modelo',
    '    best_row = metrics_df[metrics_df[\'Modelo\'] == best_model_name].iloc[0]',
    '',
    '    mod_col1, mod_col2 = st.columns([1, 2])',
    '',
    '    with mod_col1:',
    '        st.markdown(f"""',
    '        ### 🥇 {best_model_name}',
    '',
    '        **Modelo seleccionado** tras comparar 4 modelos',
    '        candidatos sobre el conjunto de prueba.',
    '',
    '        **Justificación técnica:**',
    '        El modelo fue seleccionado tras una comparación',
    '        sistemática contra Regresión Logística, Random Forest',
    '        y Logística + SMOTE, evaluados sobre el mismo conjunto',
    '        de prueba con métricas apropiadas para datos',
    '        desbalanceados. El modelo obtuvo el mayor PR-AUC,',
    '        métrica más fiable que accuracy o ROC-AUC en',
    '        escenarios con clases asimétricas como el nuestro.',
    '',
    '        **Métricas en el set de prueba:**',
    '        """)',
    '',
    '        c1, c2 = st.columns(2)',
    '        c1.metric("PR-AUC", f"{best_row[\'PR-AUC\']:.4f}")',
    '        c2.metric("ROC-AUC", f"{best_row[\'ROC-AUC\']:.4f}")',
    '        c3, c4 = st.columns(2)',
    '        c3.metric("Recall", f"{best_row[\'Recall\']:.4f}")',
    '        c4.metric("F1-Score", f"{best_row[\'F1\']:.4f}")',
    '',
    '    with mod_col2:',
    '        # Gráfico de métricas SOLO del mejor modelo',
    '        best_metrics = pd.DataFrame({',
    '            \'Métrica\': [\'Accuracy\', \'Precision\', \'Recall\', \'F1\', \'ROC-AUC\', \'PR-AUC\'],',
    '            \'Valor\': [best_row[\'Accuracy\'], best_row[\'Precision\'], best_row[\'Recall\'],',
    '                      best_row[\'F1\'], best_row[\'ROC-AUC\'], best_row[\'PR-AUC\']]',
    '        })',
    '        fig = px.bar(',
    '            best_metrics, x=\'Métrica\', y=\'Valor\',',
    '            text_auto=\'.3f\',',
    '            title=f\'Métricas del modelo ganador — {best_model_name}\',',
    '            color=\'Valor\',',
    '            color_continuous_scale=\'Greens\'',
    '        )',
    '        fig.update_layout(height=380, yaxis_range=[0, 1], coloraxis_showscale=False)',
    '        st.plotly_chart(fig, use_container_width=True)',
    '',
    '    st.info("💡 Para ver la comparativa con los otros 3 modelos candidatos, ve a la página **📈 Desempeño del modelo**.")',
    '',
    '    st.markdown("---")',
    '',
    '    # ============ TOP VARIABLES PREDICTORAS ============',
    '    st.subheader("⭐ Top variables predictoras (ranking del modelo ganador)")',
    '',
    '    # Tabla descriptiva de top variables',
    '    top_vars_data = pd.DataFrame({',
    '        \'Variable\': [\'Fault\', \'BasePolicy\', \'PastNumberOfClaims\', \'AddressChange_Claim\',',
    '                     \'PolicyType\', \'VehiclePrice\', \'AgeOfVehicle\', \'Deductible\'],',
    '        \'Tipo\': [\'Categórica\', \'Categórica\', \'Ordinal\', \'Ordinal\',',
    '                 \'Categórica\', \'Ordinal\', \'Ordinal\', \'Continua\'],',
    '        \'Importancia\': [\'Muy alta\', \'Muy alta\', \'Alta\', \'Alta\',',
    '                        \'Media-alta\', \'Media\', \'Media\', \'Media\'],',
    '        \'Interpretación\': [',
    '            \'Responsable del siniestro (Policy Holder = mayor riesgo)\',',
    '            \'Tipo de póliza base (All Perils/Collision = mayor riesgo)\',',
    '            \'Reclamaciones previas del asegurado\',',
    '            \'Cambio de dirección reciente antes de la reclamación\',',
    '            \'Combinación de tipo de póliza y categoría\',',
    '            \'Rango de precio del vehículo\',',
    '            \'Antigüedad del vehículo\',',
    '            \'Monto del deducible de la póliza\'',
    '        ]',
    '    })',
    '    st.dataframe(top_vars_data, use_container_width=True, hide_index=True)',
    '',
    '    st.markdown("---")',
    '',
    '    # ============ APLICACIÓN PRÁCTICA ============',
    '    st.subheader("💡 Aplicación práctica del modelo")',
    '',
    '    app_cols = st.columns(3)',
    '    with app_cols[0]:',
    '        st.info("""',
    '        **🟢 Probabilidad < 30%**',
    '',
    '        **Riesgo bajo.** Procesamiento automático',
    '        sin revisión adicional. Reduce carga',
    '        operativa al equipo.',
    '        """)',
    '    with app_cols[1]:',
    '        st.warning("""',
    '        **🟡 Probabilidad 30% – 60%**',
    '',
    '        **Riesgo medio.** Revisión estándar por',
    '        analista. Solicitar documentación',
    '        complementaria si aplica.',
    '        """)',
    '    with app_cols[2]:',
    '        st.error("""',
    '        **🔴 Probabilidad > 60%**',
    '',
    '        **Riesgo alto.** Investigación',
    '        profunda. Asignar a analista senior y',
    '        considerar peritaje adicional.',
    '        """)',
    '',
    '    st.markdown("---")',
    '',
    '    # ============ NAVEGACIÓN ============',
    '    st.subheader("🧭 Cómo navegar esta aplicación")',
    '    nav_cols = st.columns(3)',
    '    with nav_cols[0]:',
    '        st.markdown("""',
    '        **📊 EDA Interactivo**',
    '',
    '        Explora el dataset filtrando por',
    '        las 15 variables categóricas. Visualiza',
    '        tasas de fraude por cualquier segmento.',
    '        """)',
    '    with nav_cols[1]:',
    '        st.markdown("""',
    '        **🎯 Predicción**',
    '',
    '        Ingresa los datos de una reclamación',
    '        y obtén la probabilidad de fraude',
    '        comparada entre los 4 modelos.',
    '        """)',
    '    with nav_cols[2]:',
    '        st.markdown("""',
    '        **📈 Desempeño**',
    '',
    '        Tabla comparativa de métricas con el',
    '        mejor modelo resaltado, curvas ROC/PR',
    '        y matrices de confusión.',
    '        """)',
    '',
    '',
    '# --------------------------------------------------------------',
    '# Página 2 – EDA Interactivo',
    '# --------------------------------------------------------------',
    'elif page == "📊 EDA Interactivo":',
    '    st.header("📊 Análisis exploratorio interactivo")',
    '    st.markdown(f"**{len(FILTROS_EDA)} filtros disponibles** — selecciona los valores que quieras incluir.")',
    '',
    '    btn_col1, btn_col2, _ = st.columns([1, 1, 6])',
    '    select_all = btn_col1.button("✅ Seleccionar todo")',
    '    clear_all  = btn_col2.button("❌ Limpiar todo")',
    '    st.markdown("---")',
    '',
    '    selections = {}',
    '    cols = st.columns(3)',
    '    for i, (col_name, label) in enumerate(FILTROS_EDA.items()):',
    '        opts = sorted(df[col_name].dropna().unique().tolist())',
    '        if select_all:',
    '            default = opts',
    '        elif clear_all:',
    '            default = []',
    '        else:',
    '            default = opts',
    '        with cols[i % 3]:',
    '            selections[col_name] = st.multiselect(',
    '                label, opts, default=default, key=f"filter_{col_name}"',
    '            )',
    '',
    '    st.markdown("---")',
    '',
    '    mask = pd.Series([True] * len(df), index=df.index)',
    '    for col_name, selected in selections.items():',
    '        if selected:',
    '            mask = mask & df[col_name].isin(selected)',
    '    df_f = df[mask]',
    '',
    '    if len(df_f) == 0:',
    '        st.warning("No hay datos con los filtros seleccionados. Ajusta los filtros.")',
    '    else:',
    '        m1, m2, m3 = st.columns(3)',
    '        m1.metric("Filas filtradas", f"{len(df_f):,}")',
    '        m2.metric("Tasa de fraude", f"{df_f[\'FraudFound_P\'].mean()*100:.2f}%")',
    '        m3.metric("% del total", f"{len(df_f)/len(df)*100:.1f}%")',
    '',
    '        c1, c2 = st.columns(2)',
    '        with c1:',
    '            fig = px.histogram(',
    '                df_f, x=\'Age\', color=\'FraudFound_P\',',
    '                barmode=\'overlay\', nbins=40, opacity=0.7,',
    '                color_discrete_map={0: \'#2E86AB\', 1: \'#E63946\'},',
    '                title=\'Distribución de edad por clase\'',
    '            )',
    '            st.plotly_chart(fig, use_container_width=True)',
    '        with c2:',
    '            rate = (',
    '                df_f.groupby(\'Make\')[\'FraudFound_P\']',
    '                .mean()',
    '                .sort_values(ascending=False) * 100',
    '            )',
    '            fig = px.bar(',
    '                x=rate.index, y=rate.values,',
    '                title=\'Tasa de fraude (%) por marca\',',
    '                labels={\'x\': \'Marca\', \'y\': \'Tasa fraude (%)\'}',
    '            )',
    '            fig.add_hline(',
    '                y=df[\'FraudFound_P\'].mean() * 100,',
    '                line_dash=\'dash\', line_color=\'red\',',
    '                annotation_text=\'Promedio global\'',
    '            )',
    '            st.plotly_chart(fig, use_container_width=True)',
    '',
    '        if df_f[\'BasePolicy\'].nunique() > 0 and df_f[\'Fault\'].nunique() > 0:',
    '            st.subheader("Tasa de fraude por BasePolicy × Fault")',
    '            pivot = df_f.pivot_table(',
    '                index=\'BasePolicy\', columns=\'Fault\',',
    '                values=\'FraudFound_P\', aggfunc=\'mean\'',
    '            ).round(4) * 100',
    '            fig = px.imshow(',
    '                pivot, text_auto=\'.2f\', color_continuous_scale=\'Reds\',',
    '                title=\'Tasa de fraude (%) — BasePolicy × Fault\'',
    '            )',
    '            st.plotly_chart(fig, use_container_width=True)',
    '',
    '        st.subheader("Tasa de fraude por variable")',
    '        vars_to_plot = [c for c in FILTROS_EDA if df_f[c].nunique() > 1]',
    '        for i in range(0, len(vars_to_plot), 2):',
    '            row_cols = st.columns(2)',
    '            for j, col_name in enumerate(vars_to_plot[i:i+2]):',
    '                label = FILTROS_EDA[col_name]',
    '                tasa = (',
    '                    df_f.groupby(col_name)[\'FraudFound_P\']',
    '                    .mean()',
    '                    .sort_values(ascending=False) * 100',
    '                ).reset_index()',
    '                tasa.columns = [col_name, \'Tasa fraude (%)\']',
    '                fig = px.bar(',
    '                    tasa, x=col_name, y=\'Tasa fraude (%)\',',
    '                    title=f\'Tasa por {label}\',',
    '                    color=\'Tasa fraude (%)\',',
    '                    color_continuous_scale=\'Reds\'',
    '                )',
    '                fig.add_hline(',
    '                    y=df[\'FraudFound_P\'].mean() * 100,',
    '                    line_dash=\'dash\', line_color=\'gray\',',
    '                    annotation_text=\'Promedio\'',
    '                )',
    '                with row_cols[j]:',
    '                    st.plotly_chart(fig, use_container_width=True)',
    '',
    '# --------------------------------------------------------------',
    '# Página 3 – Predicción individual',
    '# --------------------------------------------------------------',
    'elif page == "🎯 Predicción":',
    '    st.header(f"🎯 Predicción individual — Modelo: {selected_model_name}")',
    '    st.markdown("Ingresa los datos de la reclamación:")',
    '',
    '    with st.form("pred_form"):',
    '        c1, c2, c3 = st.columns(3)',
    '        with c1:',
    '            Month = st.selectbox("Mes accidente", [',
    '                \'Jan\',\'Feb\',\'Mar\',\'Apr\',\'May\',\'Jun\',',
    '                \'Jul\',\'Aug\',\'Sep\',\'Oct\',\'Nov\',\'Dec\'',
    '            ])',
    '            DayOfWeek = st.selectbox("Día semana accidente", [',
    '                \'Monday\',\'Tuesday\',\'Wednesday\',\'Thursday\',',
    '                \'Friday\',\'Saturday\',\'Sunday\'',
    '            ])',
    '            Make = st.selectbox("Marca", sorted(df[\'Make\'].unique()))',
    '            AccidentArea = st.selectbox("Zona accidente", [\'Urban\', \'Rural\'])',
    '            Sex = st.selectbox("Sexo", [\'Male\', \'Female\'])',
    '            MaritalStatus = st.selectbox("Estado civil", [',
    '                \'Single\', \'Married\', \'Widow\', \'Divorced\'',
    '            ])',
    '            Age = st.slider("Edad del asegurado", 16, 80, 35)',
    '',
    '        with c2:',
    '            Fault = st.selectbox("Responsable", [\'Policy Holder\', \'Third Party\'])',
    '            PolicyType = st.selectbox("Tipo de póliza", sorted(df[\'PolicyType\'].unique()))',
    '            VehicleCategory = st.selectbox("Categoría vehículo", [\'Sport\', \'Sedan\', \'Utility\'])',
    '            VehiclePrice = st.selectbox(',
    '                "Rango precio vehículo",',
    '                list(ORDINAL_MAPPINGS[\'VehiclePrice\'].keys())',
    '            )',
    '            BasePolicy = st.selectbox("BasePolicy", [\'Liability\', \'Collision\', \'All Perils\'])',
    '            Deductible = st.select_slider(',
    '                "Deducible ($)", options=[300, 400, 500, 700], value=400',
    '            )',
    '            DriverRating = st.slider("DriverRating (1-4)", 1, 4, 2)',
    '',
    '        with c3:',
    '            Days_Policy_Accident = st.selectbox(',
    '                "Días póliza-accidente",',
    '                list(ORDINAL_MAPPINGS[\'Days_Policy_Accident\'].keys())',
    '            )',
    '            Days_Policy_Claim = st.selectbox(',
    '                "Días póliza-reclamación",',
    '                list(ORDINAL_MAPPINGS[\'Days_Policy_Claim\'].keys())',
    '            )',
    '            PastNumberOfClaims = st.selectbox(',
    '                "Reclamaciones previas",',
    '                list(ORDINAL_MAPPINGS[\'PastNumberOfClaims\'].keys())',
    '            )',
    '            AgeOfVehicle = st.selectbox(',
    '                "Antigüedad vehículo",',
    '                list(ORDINAL_MAPPINGS[\'AgeOfVehicle\'].keys())',
    '            )',
    '            PoliceReportFiled = st.selectbox("Reporte policial", [\'Yes\', \'No\'])',
    '            WitnessPresent = st.selectbox("Testigos presentes", [\'Yes\', \'No\'])',
    '',
    '        c4, c5 = st.columns(2)',
    '        with c4:',
    '            AgentType = st.selectbox("Tipo de agente", [\'Internal\', \'External\'])',
    '            NumberOfSuppliments = st.selectbox(',
    '                "Suplementos a la reclamación",',
    '                list(ORDINAL_MAPPINGS[\'NumberOfSuppliments\'].keys())',
    '            )',
    '        with c5:',
    '            AddressChange_Claim = st.selectbox(',
    '                "Cambio de dirección",',
    '                list(ORDINAL_MAPPINGS[\'AddressChange_Claim\'].keys())',
    '            )',
    '            NumberOfCars = st.selectbox(',
    '                "Número de vehículos",',
    '                list(ORDINAL_MAPPINGS[\'NumberOfCars\'].keys())',
    '            )',
    '',
    '        submitted = st.form_submit_button("🚀 Predecir", type="primary")',
    '',
    '    if submitted:',
    '        new = {',
    '            \'Month\': Month, \'WeekOfMonth\': 3, \'DayOfWeek\': DayOfWeek,',
    '            \'Make\': Make, \'AccidentArea\': AccidentArea,',
    '            \'DayOfWeekClaimed\': DayOfWeek, \'MonthClaimed\': Month,',
    '            \'WeekOfMonthClaimed\': 3, \'Sex\': Sex,',
    '            \'MaritalStatus\': MaritalStatus, \'Age\': Age,',
    '            \'Fault\': Fault, \'PolicyType\': PolicyType,',
    '            \'VehicleCategory\': VehicleCategory,',
    '            \'VehiclePrice\': ORDINAL_MAPPINGS[\'VehiclePrice\'][VehiclePrice],',
    '            \'RepNumber\': 12,',
    '            \'Deductible\': Deductible,',
    '            \'DriverRating\': DriverRating,',
    '            \'Days_Policy_Accident\': ORDINAL_MAPPINGS[\'Days_Policy_Accident\'][Days_Policy_Accident],',
    '            \'Days_Policy_Claim\': ORDINAL_MAPPINGS[\'Days_Policy_Claim\'][Days_Policy_Claim],',
    '            \'PastNumberOfClaims\': ORDINAL_MAPPINGS[\'PastNumberOfClaims\'][PastNumberOfClaims],',
    '            \'AgeOfVehicle\': ORDINAL_MAPPINGS[\'AgeOfVehicle\'][AgeOfVehicle],',
    '            \'PoliceReportFiled\': PoliceReportFiled,',
    '            \'WitnessPresent\': WitnessPresent,',
    '            \'AgentType\': AgentType,',
    '            \'NumberOfSuppliments\': ORDINAL_MAPPINGS[\'NumberOfSuppliments\'][NumberOfSuppliments],',
    '            \'AddressChange_Claim\': ORDINAL_MAPPINGS[\'AddressChange_Claim\'][AddressChange_Claim],',
    '            \'NumberOfCars\': ORDINAL_MAPPINGS[\'NumberOfCars\'][NumberOfCars],',
    '            \'BasePolicy\': BasePolicy',
    '        }',
    '        X_new = pd.DataFrame([new])',
    '',
    '        try:',
    '            prob = selected_model.predict_proba(X_new)[0, 1]',
    '            pred = int(prob >= 0.5)',
    '',
    '            cA, cB = st.columns(2)',
    '            cA.metric("Probabilidad de fraude", f"{prob*100:.2f}%")',
    '            cB.metric("Predicción", "🚨 FRAUDE" if pred == 1 else "✅ Legítima")',
    '',
    '            fig = go.Figure(go.Indicator(',
    '                mode="gauge+number",',
    '                value=prob * 100,',
    '                title={\'text\': f"Riesgo de fraude (%) — {selected_model_name}"},',
    '                gauge={',
    '                    \'axis\': {\'range\': [0, 100]},',
    '                    \'bar\': {\'color\': "#E63946" if prob > 0.5 else "#2E86AB"},',
    '                    \'steps\': [',
    '                        {\'range\': [0, 30],   \'color\': \'#A8E6A1\'},',
    '                        {\'range\': [30, 60],  \'color\': \'#FFE066\'},',
    '                        {\'range\': [60, 100], \'color\': \'#FFB3B3\'}',
    '                    ]',
    '                }',
    '            ))',
    '            st.plotly_chart(fig, use_container_width=True)',
    '',
    '            st.subheader("🔍 Comparativa con los 4 modelos")',
    '            comp_data = []',
    '            for name, m in models_dict.items():',
    '                p = m.predict_proba(X_new)[0, 1]',
    '                comp_data.append({',
    '                    \'Modelo\': name + (\' ⭐\' if name == best_model_name else \'\'),',
    '                    \'Probabilidad fraude (%)\': round(p * 100, 2),',
    '                    \'Predicción\': \'FRAUDE\' if p >= 0.5 else \'Legítima\'',
    '                })',
    '            comp_df = pd.DataFrame(comp_data)',
    '            st.dataframe(comp_df, use_container_width=True, hide_index=True)',
    '',
    '            risk_factors = []',
    '            if Fault == \'Policy Holder\':',
    '                risk_factors.append("El asegurado es el responsable del siniestro")',
    '            if BasePolicy in [\'Collision\', \'All Perils\']:',
    '                risk_factors.append(f"Póliza tipo {BasePolicy} con alta incidencia de fraude")',
    '            if PastNumberOfClaims in [\'2 to 4\', \'more than 4\']:',
    '                risk_factors.append("Historial de múltiples reclamaciones previas")',
    '            if AddressChange_Claim in [\'under 6 months\', \'1 year\']:',
    '                risk_factors.append("Cambio de dirección reciente antes de la reclamación")',
    '            if risk_factors:',
    '                msg = "**Factores de riesgo detectados:**\\n- " + "\\n- ".join(risk_factors)',
    '                st.warning(msg)',
    '',
    '        except Exception as e:',
    '            st.error(f"Error en la predicción: {e}")',
    '',
    '# --------------------------------------------------------------',
    '# Página 4 – Desempeño del modelo',
    '# --------------------------------------------------------------',
    'else:',
    '    st.header("📈 Desempeño y comparación de los 4 modelos")',
    '    st.markdown("Métricas calculadas sobre el mismo conjunto de prueba (20% del dataset, estratificado).")',
    '',
    '    X = df.drop(columns=[\'FraudFound_P\'])',
    '    y = df[\'FraudFound_P\']',
    '    _, X_te, _, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)',
    '',
    '    @st.cache_data',
    '    def compute_all_metrics(_models_dict, _X_te, _y_te):',
    '        rows = []',
    '        curves = {}',
    '        for name, m in _models_dict.items():',
    '            y_pred  = m.predict(_X_te)',
    '            y_proba = m.predict_proba(_X_te)[:, 1]',
    '            rows.append({',
    '                \'Modelo\':    name,',
    '                \'Accuracy\':  accuracy_score(_y_te, y_pred),',
    '                \'Precision\': precision_score(_y_te, y_pred, zero_division=0),',
    '                \'Recall\':    recall_score(_y_te, y_pred),',
    '                \'F1\':        f1_score(_y_te, y_pred),',
    '                \'ROC-AUC\':   roc_auc_score(_y_te, y_proba),',
    '                \'PR-AUC\':    average_precision_score(_y_te, y_proba),',
    '            })',
    '            fpr, tpr, _ = roc_curve(_y_te, y_proba)',
    '            prec, rec, _ = precision_recall_curve(_y_te, y_proba)',
    '            curves[name] = {',
    '                \'fpr\': fpr, \'tpr\': tpr, \'precision\': prec, \'recall\': rec,',
    '                \'roc_auc\': roc_auc_score(_y_te, y_proba),',
    '                \'pr_auc\': average_precision_score(_y_te, y_proba),',
    '                \'cm\': confusion_matrix(_y_te, y_pred)',
    '            }',
    '        return pd.DataFrame(rows), curves',
    '',
    '    metrics_df, curves = compute_all_metrics(models_dict, X_te, y_te)',
    '',
    '    best_idx = metrics_df[\'PR-AUC\'].idxmax()',
    '    best_name_auto = metrics_df.loc[best_idx, \'Modelo\']',
    '',
    '    st.success(',
    '        f"⭐ **Mejor modelo (por PR-AUC):** {best_name_auto} "',
    '        f"con PR-AUC = {metrics_df.loc[best_idx, \'PR-AUC\']:.4f}"',
    '    )',
    '',
    '    st.subheader("📊 Tabla comparativa de métricas")',
    '',
    '    def highlight_best(row):',
    '        if row[\'Modelo\'] == best_name_auto:',
    '            return [\'background-color: #C6EFCE; color: #006100; font-weight: bold\'] * len(row)',
    '        return [\'\'] * len(row)',
    '',
    '    def highlight_max(s):',
    '        if s.dtype.kind in \'biufc\':',
    '            is_max = s == s.max()',
    '            return [\'background-color: #FFEB9C; font-weight: bold\' if v else \'\' for v in is_max]',
    '        return [\'\'] * len(s)',
    '',
    '    styled = (',
    '        metrics_df.style',
    '        .apply(highlight_best, axis=1)',
    '        .apply(highlight_max, subset=[\'Accuracy\', \'Precision\', \'Recall\', \'F1\', \'ROC-AUC\', \'PR-AUC\'])',
    '        .format({',
    '            \'Accuracy\':  \'{:.4f}\',',
    '            \'Precision\': \'{:.4f}\',',
    '            \'Recall\':    \'{:.4f}\',',
    '            \'F1\':        \'{:.4f}\',',
    '            \'ROC-AUC\':   \'{:.4f}\',',
    '            \'PR-AUC\':    \'{:.4f}\',',
    '        })',
    '    )',
    '    st.dataframe(styled, use_container_width=True, hide_index=True)',
    '    st.caption(',
    '        "🟩 Verde = mejor modelo global (por PR-AUC)  |  "',
    '        "🟨 Amarillo = mejor valor por columna"',
    '    )',
    '',
    '    st.subheader("📊 Comparación visual de indicadores")',
    '    metrics_long = metrics_df.melt(',
    '        id_vars=\'Modelo\', var_name=\'Métrica\', value_name=\'Valor\'',
    '    )',
    '    fig = px.bar(',
    '        metrics_long, x=\'Métrica\', y=\'Valor\', color=\'Modelo\',',
    '        barmode=\'group\', text_auto=\'.3f\',',
    '        title=\'Métricas comparadas — los 4 modelos\',',
    '        color_discrete_sequence=px.colors.qualitative.Set2',
    '    )',
    '    fig.update_layout(height=500, yaxis_range=[0, 1])',
    '    st.plotly_chart(fig, use_container_width=True)',
    '',
    '    st.subheader("🎯 Vista radar — perfil de cada modelo")',
    '    radar_metrics = [\'Accuracy\', \'Precision\', \'Recall\', \'F1\', \'ROC-AUC\', \'PR-AUC\']',
    '    fig = go.Figure()',
    '    colors = [\'#2E86AB\', \'#E63946\', \'#06A77D\', \'#F4A261\']',
    '    for i, row in metrics_df.iterrows():',
    '        fig.add_trace(go.Scatterpolar(',
    '            r=[row[m] for m in radar_metrics],',
    '            theta=radar_metrics,',
    '            fill=\'toself\',',
    '            name=row[\'Modelo\'] + (\' ⭐\' if row[\'Modelo\'] == best_name_auto else \'\'),',
    '            line=dict(color=colors[i % len(colors)])',
    '        ))',
    '    fig.update_layout(',
    '        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),',
    '        showlegend=True, height=500,',
    '        title=\'Perfil de desempeño por modelo\'',
    '    )',
    '    st.plotly_chart(fig, use_container_width=True)',
    '',
    '    st.subheader("📈 Curvas ROC y Precision-Recall")',
    '    fig = make_subplots(rows=1, cols=2,',
    '                        subplot_titles=(\'Curva ROC\', \'Curva Precision-Recall\'))',
    '    for i, (name, c) in enumerate(curves.items()):',
    '        suffix = \' ⭐\' if name == best_name_auto else \'\'',
    '        fig.add_trace(',
    '            go.Scatter(x=c[\'fpr\'], y=c[\'tpr\'], mode=\'lines\',',
    '                       name=f"{name}{suffix} (AUC={c[\'roc_auc\']:.3f})",',
    '                       legendgroup=name,',
    '                       line=dict(color=colors[i % len(colors)], width=3)),',
    '            row=1, col=1',
    '        )',
    '        fig.add_trace(',
    '            go.Scatter(x=c[\'recall\'], y=c[\'precision\'], mode=\'lines\',',
    '                       name=f"{name} (AP={c[\'pr_auc\']:.3f})",',
    '                       legendgroup=name, showlegend=False,',
    '                       line=dict(color=colors[i % len(colors)], width=3)),',
    '            row=1, col=2',
    '        )',
    '    fig.add_trace(',
    '        go.Scatter(x=[0, 1], y=[0, 1], mode=\'lines\',',
    '                   line=dict(dash=\'dash\', color=\'gray\'), showlegend=False),',
    '        row=1, col=1',
    '    )',
    '    fig.update_xaxes(title_text=\'FPR\', row=1, col=1)',
    '    fig.update_yaxes(title_text=\'TPR\', row=1, col=1)',
    '    fig.update_xaxes(title_text=\'Recall\', row=1, col=2)',
    '    fig.update_yaxes(title_text=\'Precision\', row=1, col=2)',
    '    fig.update_layout(height=500, title_text=\'ROC y PR — los 4 modelos\')',
    '    st.plotly_chart(fig, use_container_width=True)',
    '',
    '    st.subheader("🧮 Matrices de confusión")',
    '    st.caption(',
    '        "**Convenciones:**  "',
    '        "**VP** = Verdadero Positivo · "',
    '        "**VN** = Verdadero Negativo · "',
    '        "**FP** = Falso Positivo · "',
    '        "**FN** = Falso Negativo"',
    '    )',
    '    cm_cols = st.columns(len(curves))',
    '    for i, (name, c) in enumerate(curves.items()):',
    '        with cm_cols[i]:',
    '            suffix = \' ⭐\' if name == best_name_auto else \'\'',
    '            cm_arr = c[\'cm\']',
    '            # Construir etiquetas con abreviaciones VP/VN/FP/FN',
    '            labels_text = [',
    '                [f"VN: {cm_arr[0,0]}", f"FP: {cm_arr[0,1]}"],',
    '                [f"FN: {cm_arr[1,0]}", f"VP: {cm_arr[1,1]}"]',
    '            ]',
    '            fig = go.Figure(data=go.Heatmap(',
    '                z=cm_arr,',
    '                x=[\'Pred. Legítimo\', \'Pred. Fraude\'],',
    '                y=[\'Real Legítimo\', \'Real Fraude\'],',
    '                text=labels_text,',
    '                texttemplate=\'%{text}\',',
    '                textfont={\'size\': 14, \'color\': \'black\'},',
    '                colorscale=\'Blues\',',
    '                showscale=False',
    '            ))',
    '            fig.update_layout(',
    '                title=f"{name}{suffix}",',
    '                height=380,',
    '                yaxis=dict(autorange=\'reversed\')',
    '            )',
    '            st.plotly_chart(fig, use_container_width=True)',
    '',
    '',
]

with open('app.py', 'w', encoding='utf-8') as _f:
    _f.write('\n'.join(app_lines))

import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'py_compile', 'app.py'], capture_output=True, text=True)
if result.returncode == 0:
    print(f'app.py generado correctamente ({len(app_lines)} líneas, sintaxis validada)')
else:
    print('Error de sintaxis en app.py:')
    print(result.stderr)


app.py generado correctamente (859 líneas, sintaxis validada)
